In [1]:
# !pip install mlxtend

In [2]:
import pandas as pd

#### Create the dataset

In [3]:
data = [
    ['apple', 'beer', 'rice', 'chicken'],
    ['apple', 'beer', 'rice'],
    ['apple', 'beer'],
    ['apple'],
    ['milk', 'beer', 'rice', 'chicken'],
    ['milk', 'beer', 'rice'],
    ['milk', 'beer'],
    ['milk']
]

In [4]:
data

[['apple', 'beer', 'rice', 'chicken'],
 ['apple', 'beer', 'rice'],
 ['apple', 'beer'],
 ['apple'],
 ['milk', 'beer', 'rice', 'chicken'],
 ['milk', 'beer', 'rice'],
 ['milk', 'beer'],
 ['milk']]

### Transaction Encoding

In [9]:
# !pip install mlxtend

In [10]:
from mlxtend.preprocessing import TransactionEncoder

In [11]:
te = TransactionEncoder()

In [12]:
df_enc = te.fit_transform(data)

In [13]:
df_enc

array([[ True,  True,  True, False,  True],
       [ True,  True, False, False,  True],
       [ True,  True, False, False, False],
       [ True, False, False, False, False],
       [False,  True,  True,  True,  True],
       [False,  True, False,  True,  True],
       [False,  True, False,  True, False],
       [False, False, False,  True, False]])

In [14]:
te.columns_

['apple', 'beer', 'chicken', 'milk', 'rice']

In [15]:
df = pd.DataFrame(df_enc, columns = te.columns_)

In [16]:
df

,apple,beer,chicken,milk,rice
0,True,True,True,False,True
1,True,True,False,False,True
2,True,True,False,False,False
3,True,False,False,False,False
4,False,True,True,True,True
5,False,True,False,True,True
6,False,True,False,True,False
7,False,False,False,True,False


 ### Generate frequent itemsets

In [17]:
from mlxtend.frequent_patterns import apriori

In [18]:
freq_itemset = apriori(df, min_support= 0.25, 
                       use_colnames= True)

In [19]:
freq_itemset

,support,itemsets
0,0.500,(apple)
1,0.750,(beer)
2,0.250,(chicken)
3,0.500,(milk)
4,0.500,(rice)
5,0.375,"(apple, beer)"
6,0.250,"(apple, rice)"
7,0.250,"(beer, chicken)"
8,0.375,"(beer, milk)"
9,0.500,"(rice, beer)"


### Generate association rules

In [20]:
from mlxtend.frequent_patterns import association_rules

In [21]:
rules = association_rules(freq_itemset, metric = 'confidence',
                         min_threshold= 0.50)

In [22]:
rules

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,leverage,conviction,zhangs_metric
0,(apple),(beer),0.500,0.750,0.375,0.750000,1.000000,0.0000,1.00,0.000000
1,(beer),(apple),0.750,0.500,0.375,0.500000,1.000000,0.0000,1.00,0.000000
2,(apple),(rice),0.500,0.500,0.250,0.500000,1.000000,0.0000,1.00,0.000000
3,(rice),(apple),0.500,0.500,0.250,0.500000,1.000000,0.0000,1.00,0.000000
4,(chicken),(beer),0.250,0.750,0.250,1.000000,1.333333,0.0625,inf,0.333333
5,(beer),(milk),0.750,0.500,0.375,0.500000,1.000000,0.0000,1.00,0.000000
6,(milk),(beer),0.500,0.750,0.375,0.750000,1.000000,0.0000,1.00,0.000000
7,(rice),(beer),0.500,0.750,0.500,1.000000,1.333333,0.1250,inf,0.500000
8,(beer),(rice),0.750,0.500,0.500,0.666667,1.333333,0.1250,1.50,1.000000
9,(rice),(chicken),0.500,0.250,0.250,0.500000,2.000000,0.1250,1.50,1.000000


In [23]:
rules.columns

Index(['antecedents', 'consequents', 'antecedent support',
       'consequent support', 'support', 'confidence', 'lift', 'leverage',
       'conviction', 'zhangs_metric'],
      dtype='object')

#### Extracting specific columns

In [24]:
rules = rules[['antecedents', 'consequents', 'support',
              'confidence']]

In [25]:
rules

,antecedents,consequents,support,confidence
0,(apple),(beer),0.375,0.750000
1,(beer),(apple),0.375,0.500000
2,(apple),(rice),0.250,0.500000
3,(rice),(apple),0.250,0.500000
4,(chicken),(beer),0.250,1.000000
5,(beer),(milk),0.375,0.500000
6,(milk),(beer),0.375,0.750000
7,(rice),(beer),0.500,1.000000
8,(beer),(rice),0.500,0.666667
9,(rice),(chicken),0.250,0.500000


### Extract using condition

In [26]:
rules[rules['confidence'] > 0.5]

,antecedents,consequents,support,confidence
0,(apple),(beer),0.375,0.750000
4,(chicken),(beer),0.250,1.000000
6,(milk),(beer),0.375,0.750000
7,(rice),(beer),0.500,1.000000
8,(beer),(rice),0.500,0.666667
10,(chicken),(rice),0.250,1.000000
13,"(apple, beer)",(rice),0.250,0.666667
14,"(apple, rice)",(beer),0.250,1.000000
19,"(rice, chicken)",(beer),0.250,1.000000
20,"(beer, chicken)",(rice),0.250,1.000000


In [29]:
rules[(rules['confidence'] > 0.5) & (rules['support'] > 0.25)]

,antecedents,consequents,support,confidence
0,(apple),(beer),0.375,0.750000
6,(milk),(beer),0.375,0.750000
7,(rice),(beer),0.500,1.000000
8,(beer),(rice),0.500,0.666667


### reccomendation

In [32]:
rules[rules['antecedents'] == {'apple'}]

,antecedents,consequents,support,confidence
0,(apple),(beer),0.375,0.75
2,(apple),(rice),0.250,0.50
16,(apple),"(rice, beer)",0.250,0.50


In [33]:
rules[rules['antecedents'] == {'rice'}]

,antecedents,consequents,support,confidence
3,(rice),(apple),0.25,0.5
7,(rice),(beer),0.50,1.0
9,(rice),(chicken),0.25,0.5
11,(rice),(milk),0.25,0.5
17,(rice),"(apple, beer)",0.25,0.5
21,(rice),"(beer, chicken)",0.25,0.5
26,(rice),"(beer, milk)",0.25,0.5


In [39]:
rules[(rules['antecedents'] == {'rice'}) 
      & (rules['confidence'] > 0.5)]

,antecedents,consequents,support,confidence
7,(rice),(beer),0.5,1.0


In [36]:
rules[(rules['antecedents'] == {'rice'})
      & (rules['confidence'] > 0.5)]['consequents']

7    (beer)
Name: consequents, dtype: object

In [37]:
rules[(rules['antecedents'] == {'beer', 'milk'})
      & (rules['confidence'] > 0.5)]['consequents']

25    (rice)
Name: consequents, dtype: object

### Sort the values

In [40]:
rules.sort_values(by = 'confidence', ascending= False);

In [41]:
rules.to_csv('rules.csv', index = False)